In [ ]:
import pandas as pd
import re
from collections import defaultdict



In [ ]:
# -----------------------------
# 1. Mots-clés à filtrer (correspondance EXACTE)
# -----------------------------
keywords = [
    "geographie",
    "mathematik",
    "physik",
    "data science",
    "data analytics",
    "informatik"
]



In [ ]:
# -----------------------------
# 2. Fichiers Excel à traiter
# -----------------------------
fichiers = [
    "JLU-Studierendenstatistik_gesamt_SoSe_2025_Köpfe.xlsx",
    "JLU-Studierendenstatistik_gesamt_WiSe_2024_Köpfe.xlsx",
    "JLU-Studierendenstatistik_gesamt_WiSe_2025_Köpfe.xlsx"
]



In [ ]:
# -----------------------------
# 3. Feuilles à traiter (les 10 avec la même structure)
# -----------------------------
feuilles_cibles = [
    "Gesamt mB Köpfe",
    "Gesamt oB Köpfe",
    "Studierende in RSZ",
    "1. Fachsemester Köpfe",
    "1. Hochschulsemester Köpfe",
    "weibliche Studierende Köpfe",
    "männliche Studierende Köpfe",
    "ausländische Studierende Köpfe",
    "BildungsausländerInnen",
    "Beurlaubte"
]



In [ ]:
# -----------------------------
# 4. Noms des colonnes par position
# -----------------------------
noms_colonnes = [
    "Ebene",
    "Art_Ebene",
    "Studiengang",
    "Summe_Koepfe",
    "Bachelor_HF",
    "Bachelor_NF",
    "Master_HF",
    "Master_NF",
    "Dipl_Stx",
    "Ausl_Ab",
    "Lehramt_Prim",
    "Lehramt_SekI",
    "Lehramt_SekII",
    "Andere"
]

colonnes_a_supprimer = ["Ebene", "Art_Ebene"]



In [ ]:
# -----------------------------
# 5. Extraire semestre + année
# -----------------------------
def extraire_semestre_annee(nom_fichier):
    semestre = "SoSe" if "SoSe" in nom_fichier else "WiSe"
    annee = re.findall(r"\d{4}", nom_fichier)[0]
    return semestre, annee



In [ ]:
# -----------------------------
# 6. Traitement principal
# -----------------------------
fusion = defaultdict(list)

for fichier in fichiers:
    semestre, annee = extraire_semestre_annee(fichier)
    print(f"\n{'='*60}")
    print(f"📂 Fichier : {fichier} ({semestre} {annee})")

    try:
        excel = pd.ExcelFile(fichier)
    except FileNotFoundError:
        print(f"  ❌ Fichier introuvable !")
        continue

    for feuille in excel.sheet_names:
        feuille_clean = feuille.strip()

        if feuille_clean not in feuilles_cibles:
            print(f"  ⏭ Ignorée : {feuille_clean}")
            continue

        print(f"  ➡ Traitement : {feuille_clean}")

        df = pd.read_excel(excel, sheet_name=feuille, header=3)

        nb_cols = len(df.columns)
        df.columns = noms_colonnes[:nb_cols]

        df = df.drop(columns=[c for c in colonnes_a_supprimer if c in df.columns])

        df["Semester"] = semestre
        df["Jahr"] = annee

        # Normaliser
        df["Studiengang"] = df["Studiengang"].fillna("").astype(str).str.lower().str.strip()

        # Filtrage EXACT
        mask = df["Studiengang"].isin(keywords)
        df_filtre = df[mask]  # ← ligne manquante dans ton code !

        print(f"     ✔ {len(df_filtre)} lignes conservées")

        fusion[feuille_clean].append(df_filtre)



In [ ]:
# -----------------------------
# 7. Export : 1 CSV par feuille fusionnée
# -----------------------------
print(f"\n{'='*60}")
print("📊 Export des CSV fusionnés :")

for feuille, liste_df in fusion.items():
    df_final = pd.concat(liste_df, ignore_index=True)
    cols_numeriques = ["Summe_Koepfe", "Bachelor_HF", "Bachelor_NF", "Master_HF", "Master_NF",
                       "Dipl_Stx", "Ausl_Ab", "Lehramt_Prim", "Lehramt_SekI", "Lehramt_SekII", "Andere"]
    df_final[cols_numeriques] = df_final[cols_numeriques].fillna(0)
    nom_csv = feuille.replace(' ', '_').replace('.', '') + ".csv"
    df_final.to_csv(nom_csv, index=False, encoding="utf-8-sig")
    print(f"  ✔ {nom_csv} → {len(df_final)} lignes")

print(f"\n🎉 Terminé : {len(fusion)} fichiers CSV générés.")